# Epic 9.02 — Domain Transfer: USM to RGB and Mask-to-Image

This notebook demonstrates:
1. USM to RGB domain transfer using colormaps
2. Mask-to-image conditioned generation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from udm_epic9.rendering.usm_renderer import generate_synthetic_usm_with_cracks
from udm_epic9.domain_transfer.usm_to_rgb import USMtoRGBTransfer, mask_to_image
from udm_epic9.models.crack_types import die_crack, mold_crack

## USM to RGB Transfer

The `USMtoRGBTransfer` class converts grayscale USM images to pseudo-RGB
using physically-motivated colormaps. Different colormaps emphasise different
features of the acoustic signal.

In [ ]:
rng = np.random.default_rng(42)
usm_img, mask, meta = generate_synthetic_usm_with_cracks(
    height=256, width=256, n_cracks=2, rng=rng,
)

colormaps = ['inferno', 'jet', 'hot', 'viridis', 'plasma', 'turbo']

fig, axes = plt.subplots(2, len(colormaps) // 2 + 1, figsize=(16, 8))
axes = axes.flatten()

axes[0].imshow(usm_img, cmap='gray', vmin=0, vmax=1)
axes[0].set_title('Original USM')
axes[0].axis('off')

for i, cmap_name in enumerate(colormaps):
    transfer = USMtoRGBTransfer(method='colormap', colormap=cmap_name)
    rgb = transfer.transfer_with_cracks(usm_img, mask)
    axes[i + 1].imshow(rgb)
    axes[i + 1].set_title(f'{cmap_name}')
    axes[i + 1].axis('off')

plt.suptitle('USM -> RGB Domain Transfer (various colormaps)', fontsize=14)
plt.tight_layout()
plt.show()

## Mask-to-Image Generation

Given only a binary crack mask, `mask_to_image` generates a complete
synthetic image in either USM or RGB domain.

In [ ]:
# Create crack masks from different types
rng = np.random.default_rng(7)
die_mask, _ = die_crack(256, 256, rng=rng)
mold_mask, _ = mold_crack(256, 256, rng=rng)

fig, axes = plt.subplots(2, 3, figsize=(12, 8))

for row, (mask, name) in enumerate([(die_mask, 'Die Crack'), (mold_mask, 'Mold Crack')]):
    axes[row, 0].imshow(mask, cmap='gray')
    axes[row, 0].set_title(f'{name} Mask')
    
    usm_out = mask_to_image(mask, target_domain='usm')
    axes[row, 1].imshow(usm_out, cmap='gray', vmin=0, vmax=1)
    axes[row, 1].set_title('Mask -> USM')
    
    rgb_out = mask_to_image(mask, target_domain='rgb')
    axes[row, 2].imshow(rgb_out)
    axes[row, 2].set_title('Mask -> RGB')

for ax in axes.flatten(): ax.axis('off')
plt.suptitle('Mask-to-Image Generation', fontsize=14)
plt.tight_layout()
plt.show()

## Diversity of Generated Images

Each call to `mask_to_image` with the same mask produces a different
background, demonstrating the stochastic nature of the generation.

In [ ]:
mask = die_mask

fig, axes = plt.subplots(1, 5, figsize=(20, 4))
axes[0].imshow(mask, cmap='gray')
axes[0].set_title('Input Mask')
axes[0].axis('off')

for i in range(4):
    rng_i = np.random.default_rng(i * 100)
    rgb = mask_to_image(mask, target_domain='rgb', rng=rng_i)
    axes[i + 1].imshow(rgb)
    axes[i + 1].set_title(f'Generated #{i+1}')
    axes[i + 1].axis('off')

plt.suptitle('Same Mask, Different Backgrounds', fontsize=14)
plt.tight_layout()
plt.show()